In [ ]:
# Code for CNN AND RESNET18 Classifiers
# Safe remount — always run this first
import os, shutil, subprocess
from google.colab import drive

subprocess.run(['fusermount', '-uz', '/content/drive'], capture_output=True)
shutil.rmtree('/content/drive', ignore_errors=True)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch torchvision scikit-learn -q

import os
import re
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from collections import Counter
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [ ]:
# ── Config ────
HEALTHY_DIR    = '/content/drive/MyDrive/Dissertation2026/coronal-healthy'
ALZ_REAL_DIR   = '/content/drive/MyDrive/Dissertation2026/oasis-coronal-alzheimers'
ALZ_SYNTH_DIR = '/content/drive/MyDrive/Dissertation2026/ddpm-coronal-v3_output/fid_generated_350'  # 350 DDPM synthetic images
# ─────────────────────────────────────────────────────────

IMAGE_SIZE    = 128
BATCH_SIZE    = 16
NUM_EPOCHS    = 20
LEARNING_RATE = 1e-4
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Quick verification
print(f"Healthy images   : {len(os.listdir(HEALTHY_DIR))}")
print(f"Real Alzheimer's : {len(os.listdir(ALZ_REAL_DIR))}")
print(f"Synthetic        : {len(os.listdir(ALZ_SYNTH_DIR))}")

Healthy images   : 2352
Real Alzheimer's : 700
Synthetic        : 350


In [ ]:
def get_subject_id(filename):
    match = re.match(r'(OAS1_\d+_MR\d+)', filename)
    return match.group(1) if match else None

# Quick test — should print a subject ID
print(get_subject_id(os.listdir(HEALTHY_DIR)[0]))

OAS1_0258_MR1


In [ ]:
# Data splitting
def split_by_subject(directory, test_size=0.2, seed=42):                    # code refined by Claude AI
    files = [f for f in os.listdir(directory) if f.endswith('.png')]
    subject_files = {}
    for f in files:
        sid = get_subject_id(f)
        subject_files.setdefault(sid, []).append(f)
    subjects = list(subject_files.keys())
    train_subs, test_subs = train_test_split(
        subjects, test_size=test_size, random_state=seed
    )
    train_files = [f for s in train_subs for f in subject_files[s]]
    test_files  = [f for s in test_subs  for f in subject_files[s]]
    return train_files, test_files

healthy_train, healthy_test = split_by_subject(HEALTHY_DIR)
alz_train,     alz_test     = split_by_subject(ALZ_REAL_DIR)
synth_files = [f for f in os.listdir(ALZ_SYNTH_DIR) if f.endswith('.png')]

print(f"Healthy   — train: {len(healthy_train)}, test: {len(healthy_test)}")
print(f"Alzheimer — train: {len(alz_train)}, test: {len(alz_test)}")
print(f"Synthetic — train only: {len(synth_files)}")

Healthy   — train: 1876, test: 476
Alzheimer — train: 560, test: 140
Synthetic — train only: 350


In [ ]:
transform = transforms.Compose([                                              # code refined by Claude AI
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

class MRIClassifierDataset(Dataset):
    def __init__(self, healthy_files, healthy_dir,
                 alz_files, alz_dir, transform=None):
        self.samples = []
        for f in healthy_files:
            self.samples.append((os.path.join(healthy_dir, f), 0))
        for f in alz_files:
            self.samples.append((os.path.join(alz_dir, f), 1))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

class CombinedDataset(Dataset):
    def __init__(self, healthy_files, healthy_dir,
                 alz_real_files, alz_real_dir,
                 alz_synth_files, alz_synth_dir, transform=None):
        self.samples = []
        for f in healthy_files:
            self.samples.append((os.path.join(healthy_dir, f), 0))
        for f in alz_real_files:
            self.samples.append((os.path.join(alz_real_dir, f), 1))
        for f in alz_synth_files:
            self.samples.append((os.path.join(alz_synth_dir, f), 1))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

# Fixed test set — real data only, same for all experiments
test_dataset = MRIClassifierDataset(
    healthy_test, HEALTHY_DIR,
    alz_test, ALZ_REAL_DIR,
    transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Test set: {len(test_dataset)} images")

Test set: 616 images


In [ ]:
dataset_A = MRIClassifierDataset(
    healthy_train, HEALTHY_DIR,
    alz_train, ALZ_REAL_DIR,
    transform=transform
)
dataset_B = CombinedDataset(
    healthy_train, HEALTHY_DIR,
    alz_train, ALZ_REAL_DIR,
    synth_files, ALZ_SYNTH_DIR,
    transform=transform
)
dataset_C = MRIClassifierDataset(
    healthy_train, HEALTHY_DIR,
    synth_files, ALZ_SYNTH_DIR,
    transform=transform
)

print(f"Config A (Real only)       : {len(dataset_A)}")
print(f"Config B (Real+Synthetic)  : {len(dataset_B)}")
print(f"Config C (Synthetic only)  : {len(dataset_C)}")

Config A (Real only)       : 2436
Config B (Real+Synthetic)  : 2786
Config C (Synthetic only)  : 2226


In [ ]:
# ── Model 1: Simple CNN (same as before) ──────────────────
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.fc(self.conv(x))

# ── Model 2: ResNet18 with ImageNet pretrained weights ─────
class ResNet18Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(weights='IMAGENET1K_V1')
        # Adapt for grayscale — change first conv from 3 channels to 1
        self.model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        # Replace final layer for binary classification
        self.model.fc = nn.Linear(512, 2)

    def forward(self, x):
        return self.model(x)

print("Models defined ✅")

Models defined ✅


In [ ]:
def get_class_weights(dataset):                                                 # code refined by Claude AI
    labels = [label for _, label in dataset.samples]
    counts = Counter(labels)
    total  = len(labels)
    w0 = total / (2 * counts[0])
    w1 = total / (2 * counts[1])
    return torch.tensor([w0, w1], dtype=torch.float32).to(device)

def train_model(train_dataset, model, name, lr=LEARNING_RATE):
    print(f"\n{'='*50}\nTraining: {name}\n{'='*50}")
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    class_weights = get_class_weights(train_dataset)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"Class weights — healthy: {class_weights[0]:.3f}, alzheimer: {class_weights[1]:.3f}")

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {epoch_loss/len(loader):.4f}")
    return model

def evaluate_model(model, test_loader, name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            preds = torch.argmax(model(images.to(device)), dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels, all_preds, zero_division=0)
    f1   = f1_score(all_labels, all_preds, zero_division=0)
    cm   = confusion_matrix(all_labels, all_preds)

    print(f"\n--- {name} ---")
    print(f"Accuracy : {acc:.3f}  Precision: {prec:.3f}")
    print(f"Recall   : {rec:.3f}  F1       : {f1:.3f}")
    print(f"Confusion Matrix:\n{cm}")
    return {'name': name, 'accuracy': acc,
            'precision': prec, 'recall': rec, 'f1': f1}

In [ ]:
results = []

# ── SimpleCNN ─────────────
cnn_a = train_model(dataset_A, SimpleCNN().to(device), "SimpleCNN Config A: Real Only")
results.append(evaluate_model(cnn_a, test_loader, "SimpleCNN Config A"))

cnn_b = train_model(dataset_B, SimpleCNN().to(device), "SimpleCNN Config B: Real + Synthetic")
results.append(evaluate_model(cnn_b, test_loader, "SimpleCNN Config B"))

cnn_c = train_model(dataset_C, SimpleCNN().to(device), "SimpleCNN Config C: Synthetic Only")
results.append(evaluate_model(cnn_c, test_loader, "SimpleCNN Config C"))

# ── ResNet18  ──────────
rn_a = train_model(dataset_A, ResNet18Classifier().to(device),
                   "ResNet18 Config A: Real Only", lr=1e-5)
results.append(evaluate_model(rn_a, test_loader, "ResNet18 Config A"))

rn_b = train_model(dataset_B, ResNet18Classifier().to(device),
                   "ResNet18 Config B: Real + Synthetic", lr=1e-5)
results.append(evaluate_model(rn_b, test_loader, "ResNet18 Config B"))

rn_c = train_model(dataset_C, ResNet18Classifier().to(device),
                   "ResNet18 Config C: Synthetic Only", lr=1e-5)
results.append(evaluate_model(rn_c, test_loader, "ResNet18 Config C"))


Training: SimpleCNN Config A: Real Only
Class weights — healthy: 0.649, alzheimer: 2.175
  Epoch 5/20 | Loss: 0.3013
  Epoch 10/20 | Loss: 0.0931
  Epoch 15/20 | Loss: 0.0130
  Epoch 20/20 | Loss: 0.0074

--- SimpleCNN Config A ---
Accuracy : 0.857  Precision: 0.660
Recall   : 0.764  F1       : 0.709
Confusion Matrix:
[[421  55]
 [ 33 107]]

Training: SimpleCNN Config B: Real + Synthetic
Class weights — healthy: 0.743, alzheimer: 1.531
  Epoch 5/20 | Loss: 0.2011
  Epoch 10/20 | Loss: 0.0377
  Epoch 15/20 | Loss: 0.0051
  Epoch 20/20 | Loss: 0.0022

--- SimpleCNN Config B ---
Accuracy : 0.852  Precision: 0.676
Recall   : 0.671  F1       : 0.674
Confusion Matrix:
[[431  45]
 [ 46  94]]

Training: SimpleCNN Config C: Synthetic Only
Class weights — healthy: 0.593, alzheimer: 3.180
  Epoch 5/20 | Loss: 0.0000
  Epoch 10/20 | Loss: 0.0000
  Epoch 15/20 | Loss: 0.0000
  Epoch 20/20 | Loss: 0.0000

--- SimpleCNN Config C ---
Accuracy : 0.773  Precision: 0.000
Recall   : 0.000  F1       : 0.0

100%|██████████| 44.7M/44.7M [00:00<00:00, 218MB/s]



Training: ResNet18 Config A: Real Only
Class weights — healthy: 0.649, alzheimer: 2.175
  Epoch 5/20 | Loss: 0.1174
  Epoch 10/20 | Loss: 0.0339
  Epoch 15/20 | Loss: 0.0250
  Epoch 20/20 | Loss: 0.0221

--- ResNet18 Config A ---
Accuracy : 0.841  Precision: 0.640
Recall   : 0.686  F1       : 0.662
Confusion Matrix:
[[422  54]
 [ 44  96]]

Training: ResNet18 Config B: Real + Synthetic
Class weights — healthy: 0.743, alzheimer: 1.531
  Epoch 5/20 | Loss: 0.1199
  Epoch 10/20 | Loss: 0.0306
  Epoch 15/20 | Loss: 0.0300
  Epoch 20/20 | Loss: 0.0103

--- ResNet18 Config B ---
Accuracy : 0.851  Precision: 0.676
Recall   : 0.657  F1       : 0.667
Confusion Matrix:
[[432  44]
 [ 48  92]]

Training: ResNet18 Config C: Synthetic Only
Class weights — healthy: 0.593, alzheimer: 3.180
  Epoch 5/20 | Loss: 0.0088
  Epoch 10/20 | Loss: 0.0019
  Epoch 15/20 | Loss: 0.0011
  Epoch 20/20 | Loss: 0.0004

--- ResNet18 Config C ---
Accuracy : 0.773  Precision: 0.000
Recall   : 0.000  F1       : 0.000
Con

In [ ]:
summary = pd.DataFrame(results)
summary = summary[['name', 'accuracy', 'precision', 'recall', 'f1']]
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(summary.to_string(index=False))

summary.to_csv(
    '/content/drive/MyDrive/ddpm_coronal_output/final_all_results.csv',
    index=False
)
print("\n✅ Saved to Drive")


FINAL RESULTS SUMMARY
              name  accuracy  precision   recall       f1
SimpleCNN Config A  0.857143   0.660494 0.764286 0.708609
SimpleCNN Config B  0.852273   0.676259 0.671429 0.673835
SimpleCNN Config C  0.772727   0.000000 0.000000 0.000000
 ResNet18 Config A  0.840909   0.640000 0.685714 0.662069
 ResNet18 Config B  0.850649   0.676471 0.657143 0.666667
 ResNet18 Config C  0.772727   0.000000 0.000000 0.000000

✅ Saved to Drive
